---

#**Project Name: Utilizing Image Processing Model into ERP Systems for Product Searching in Fashion E-Commerce Platform**
# Group 7 - Research
Programming Language: Pytorch

Applied Model: ResNet50

Model Developers: Ngọc Quỳnh, Anh Hào

Data Annotators: Nhật Huyền, Uy Minh

---


# **Giai đoạn 2: Xây dựng và Fine-tune mô hình trên Colab**



**2.1. Kết nối Drive và giải nén**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Giải nén dataset (thay đường dẫn đến file zip của bạn)
!unzip /content/drive/MyDrive/dataset.zip -d /content/

: 

**2.2. Khai báo mô hình ResNet50 và Fine-tune**

Huấn luyện mô hình nhận diện các loại quần áo để nó hiểu "đặc trưng" của thời trang.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# 1. Tiền xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load dataset từ thư mục đã giải nén
train_dataset = datasets.ImageFolder('/content/dataset_fashion', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. Khởi tạo ResNet50
model = models.resnet50(pretrained=True)

# Fine-tune: Thay đổi lớp cuối cùng để khớp với số lượng phân loại của bạn
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(num_ftrs, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 4. Huấn luyện sơ bộ (Training)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(5): # Chạy 5-10 epoch tùy độ lớn dataset
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} hoàn tất. Loss: {loss.item()}")

# Lưu lại mô hình sau khi huấn luyện
torch.save(model.state_dict(), 'fashion_resnet50.pth')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 hoàn tất. Loss: 0.5171436071395874
Epoch 2 hoàn tất. Loss: 0.08835315704345703
Epoch 3 hoàn tất. Loss: 0.04990880563855171
Epoch 4 hoàn tất. Loss: 0.012623308226466179
Epoch 5 hoàn tất. Loss: 0.08734317868947983


# **Giai đoạn 3: Trích xuất Feature và Tạo Vector Database**

Sau khi có mô hình đã hiểu về thời trang, ta dùng nó để chuyển toàn bộ kho ảnh sản phẩm thành các vector.

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.0 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

# Bỏ lớp FC cuối để lấy vector đặc trưng (2048 chiều)
feature_model = torch.nn.Sequential(*(list(model.children())[:-1]))
feature_model.eval()

def extract_vector(img_path):
    from PIL import Image
    img = Image.open(img_path).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feature = feature_model(img)
    return feature.cpu().numpy().flatten()

# Duyệt toàn bộ ảnh trong kho để tạo database vector
image_list = [] # Lưu đường dẫn ảnh
vectors = []    # Lưu vector tương ứng

for root, dirs, files in os.walk('/content/dataset_fashion'):
    for file in files:
        if file.endswith(('.jpg', '.png', '.jpeg')):
            path = os.path.join(root, file)
            vectors.append(extract_vector(path))
            image_list.append(file.split('.')[0]) # Lưu Product ID từ tên file

# Tạo Index FAISS
dimension = 2048
index = faiss.IndexFlatL2(dimension)
index.add(np.array(vectors).astype('float32'))

# Lưu lại index để dùng cho API
faiss.write_index(index, "vector_db.index")
np.save("product_ids.npy", image_list)

NameError: name 'torch' is not defined

# **Giai đoạn 4: Tạo API bằng FastAPI và Ngrok**
Đây là phần kết nối Colab với Backend bên ngoài.

In [ ]:
!pip install fastapi uvicorn pyngrok nest_asyncio python-multipart

from fastapi import FastAPI, UploadFile, File
import uvicorn
from pyngrok import ngrok
import nest_asyncio
from PIL import Image
import io

app = FastAPI()

@app.post("/search")
async def search_product(file: UploadFile = File(...)):
    # 1. Đọc ảnh gửi lên từ Website
    contents = await file.read()
    img = Image.open(io.BytesIO(contents)).convert('RGB')

    # 2. Trích xuất vector của ảnh tìm kiếm
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        query_vector = feature_model(img_t).cpu().numpy().astype('float32')

    # 3. Tìm 10 sản phẩm giống nhất trong FAISS
    D, I = index.search(query_vector.reshape(1, -1), k=10)

    # 4. Trả về Product ID cho Backend
    res_ids = [image_list[i] for i in I[0]]
    return {"matched_product_ids": res_ids}

# Chạy Server API
auth_token = "YOUR_NGROK_AUTH_TOKEN" # Lấy trên ngrok.com
ngrok.set_auth_token(auth_token)
public_url = ngrok.connect(8000)
print(f"API đang chạy tại: {public_url}")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

ERROR:pyngrok.process.ngrok:t=2026-05-14T11:16:20+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-14T11:16:20+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.